In [ ]:
import os, sys, math, random, numpy as np
print('Python:', sys.version.split()[0])
# Ensure repo root is on sys.path
repo_root = os.getcwd()
if os.path.basename(repo_root) == 'notebooks':
    repo_root = os.path.abspath(os.path.join(repo_root, '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
try:
    import torch
    print('Torch:', torch.__version__)
except Exception as e:
    print('Torch import failed:', e)
np.random.seed(0); random.seed(0)

In [ ]:
import numpy as np
import math
import itertools

# --- 1. KONFIGURACJA MODELU (Castaway MDP 3.3) ---
M = 10           # Pojemność magazynu (0..10)
H_MAX = 5        # Maksymalne zdrowie (0..5, gdzie 0=śmierć)
D_MAX = 30       # Limit dni (gwarantowany ratunek w dniu 29->30)

# Nagrody
R_RES = 50.0     # Nagroda za ratunek
R_DEAD = -100.0  # Kara za śmierć

# Wymiary przestrzeni stanów
N_F = M + 1      # 11 stanów (0..10)
N_W = M + 1      # 11 stanów (0..10)
N_H = H_MAX + 1  # 6 stanów (0..5)
N_D = D_MAX + 1  # 31 stanów (0..30)

# Kodowanie stanu: s = f + N_F * (w + N_W * (h + N_H * d))
# Pre-compute strides for faster encoding/decoding
_STRIDE_F = 1
_STRIDE_W = N_F
_STRIDE_H = N_F * N_W
_STRIDE_D = N_F * N_W * N_H

def encode(f, w, h, d):
    """Optimized state encoding using pre-computed strides."""
    return int(f * _STRIDE_F + w * _STRIDE_W + h * _STRIDE_H + d * _STRIDE_D)

def decode(s):
    """Optimized state decoding using pre-computed strides."""
    s = int(s)
    d = s // _STRIDE_D
    rem = s % _STRIDE_D
    h = rem // _STRIDE_H
    rem = rem % _STRIDE_H
    w = rem // _STRIDE_W
    f = rem % _STRIDE_W
    return f, w, h, d

# Vectorized decode for batch operations
def decode_batch(states):
    """Vectorized state decoding for batches.
    
    Args:
        states: array of state indices [B]
    
    Returns:
        f, w, h, d: arrays of shape [B] each
    """
    states = np.asarray(states, dtype=np.int64)
    d = states // _STRIDE_D
    rem = states % _STRIDE_D
    h = rem // _STRIDE_H
    rem = rem % _STRIDE_H
    w = rem // _STRIDE_W
    f = rem % _STRIDE_W
    return f, w, h, d

NUM_STATES = N_F * N_W * N_H * N_D

# --- 2. PRZESTRZEŃ AKCJI ---
# Akcja to krotka: (tryb, racja_jedzenia, racja_wody)
# Tryby: 'wait', 'hunt'
# Racje: 0=None, 1=Half, 2=Full
MODES = ['wait', 'hunt']
RATIONS = [0, 1, 2]

ACTIONS = list(itertools.product(MODES, RATIONS, RATIONS))
NUM_ACTIONS = len(ACTIONS)

# Pomocnicze mapowanie dla czytelności
def get_action_desc(idx):
    m, rf, rw = ACTIONS[idx]
    return f"Mode: {m}, Eat: {rf}, Drink: {rw}"

# --- 3. DYNAMIKA PRZEJŚĆ ---
def sample_transition(state, action_idx):
    """
    Zwraca: (next_state, reward, done)
    """
    f, w, h, d = decode(state)
    
    # Sprawdzenie stanów terminalnych na wejściu (zabezpieczenie)
    if h == 0 or d >= D_MAX:
        return state, 0.0, True

    mode, target_rf, target_rw = ACTIONS[action_idx]
    
    # --- FAZA A: POGODA (Deszcz) ---
    # P(Rain)=0.25, zysk +2 wody
    if np.random.rand() < 0.25:
        w = min(M, w + 2)
    # (w przeciwnym razie w bez zmian)

    # --- FAZA B: WYPRAWA (Polowanie) ---
    if mode == 'hunt':
        # Sprawdzenie kosztu (inwestycji)
        if f >= 1 and w >= 1:
            # Ponosimy koszt
            f -= 1
            w -= 1
            
            # Szansa na sukces: 0.3 + 0.08 * h
            p_succ = 0.3 + 0.08 * h
            if np.random.rand() < p_succ:
                # Sukces: +3 jedzenia (netto +2, bo koszt -1)
                f = min(M, f + 3)
            # Porażka: brak zysku (netto -1 za koszt)
        else:
            # Nie stać nas na polowanie -> wymuszone czekanie
            pass 

    # --- FAZA C: KONSUMPCJA I ZDROWIE ---
    # Faktyczne spożycie ograniczone przez to, co mamy w magazynie
    cf = min(target_rf, f)
    cw = min(target_rw, w)
    
    # Aktualizacja magazynu
    f -= cf
    w -= cw
    
    # Zmiana zdrowia
    if cf >= 2 and cw >= 2:
        delta_h = 1   # Regeneracja
    elif cf >= 1 and cw >= 1:
        delta_h = 0   # Utrzymanie
    elif cf == 0 and cw == 0:
        delta_h = -2  # Krytyczne wycieńczenie
    else:
        delta_h = -1  # Głód lub pragnienie (jedno z nich jest 0)
        
    h = max(0, min(H_MAX, h + delta_h))

    # Obliczanie nagrody bieżącej (użyteczność)
    reward = math.log1p(cf) + math.log1p(cw)
    
    # Sprawdzenie śmierci
    if h == 0:
        return encode(f, w, h, d + 1), reward + R_DEAD, True

    # --- FAZA D: RATUNEK (Funkcja Hazardu) ---
    # Prawdopodobieństwo ratunku h(d)
    # Jeśli d = d_max - 1 (czyli 29), szansa jest 100% (ratunek w dniu 30)
    p_rescue = 1.0 if d == (D_MAX - 1) else 0.05
    
    if np.random.rand() < p_rescue:
        # Ratunek!
        # Zwracamy stan z d+1, ale flagujemy done i dodajemy bonus
        return encode(f, w, h, d + 1), reward + R_RES, True

    # --- KONIEC TURY ---
    next_d = d + 1
    next_state = encode(f, w, h, next_d)
    
    return next_state, reward, False

print(f"Castaway MDP configured: {NUM_STATES} states, {NUM_ACTIONS} actions")
print(f"State space: F∈[0,{M}], W∈[0,{M}], H∈[0,{H_MAX}], D∈[0,{D_MAX}]")



In [ ]:
# NumPy QH Q-Learning: dłuższe uczenie z bardzo powolnym spadkiem kroków
from src.algorithms.qh_qlearning import QHQLearning, train_qh_qlearning_sweep

agent = QHQLearning(
    n_states=NUM_STATES, 
    n_actions=NUM_ACTIONS,
    alpha=0.8, 
    beta=0.95,
    theta_step=0.1,      # Bazowa wartość kroku wolnego
    eta_step=0.15,       # Bazowa wartość kroku szybkiego
    theta_power=0.55,    # Bardzo powolny spadek (minimalny powyżej 0.5)
    eta_power=0.52,      # Bardzo powolny spadek
    step_offset=5000.0,  # Duży offset sprawia, że kroki długo pozostają wysokie
    init_value=0.0
)

def sampler_fn(s, a):
    return sample_transition(int(s), int(a))

print(f"Training on {NUM_STATES} states, {NUM_ACTIONS} actions")
print(f"Step sizes: eta_0 ≈ {agent._eta_schedule(1):.6f}, theta_0 ≈ {agent._theta_schedule(1):.6f}")
print(f"After 100 sweeps: eta_100 ≈ {agent._eta_schedule(100):.6f}, theta_100 ≈ {agent._theta_schedule(100):.6f}\n")

res = train_qh_qlearning_sweep(sampler_fn, agent, n_iterations=100)

print(f'\nIterations completed: {res["iteration"]}')
print(f'Sample W values for state 0 (first 6 actions):\n{agent.W[0][:6]}')
print(f'\nSample Q values for state 0 (first 6 actions):\n{agent.Q[0][:6]}')

# Znajdź najlepsze akcje dla stanu startowego (f=5, w=5, h=3, d=0)
start_state = encode(5, 5, 3, 0)
best_action_idx = np.argmax(agent.Q[start_state])
print(f'\nBest action for start (f=5, w=5, h=3, d=0): {get_action_desc(best_action_idx)}')

In [ ]:
# Torch QH Q-Learning (batched): quick single-batch update if torch is available
try:
    from src.algorithms.qh_qlearning_torch import QHQLearningTorch
    import torch
    torch.manual_seed(0)
    t_agent = QHQLearningTorch(n_states=NUM_STATES, n_actions=NUM_ACTIONS,
                               alpha=0.8, beta=0.95,
                               theta_step=0.2, eta_step=0.2)
    B = 64
    states = np.random.randint(0, NUM_STATES, size=B)
    actions = np.random.randint(0, NUM_ACTIONS, size=B)
    next_states, rewards, dones = [], [], []
    for s, a in zip(states, actions):
        ns, r, d = sample_transition(int(s), int(a))
        next_states.append(ns); rewards.append(r); dones.append(d)
    states_t = torch.tensor(states, dtype=torch.long)
    actions_t = torch.tensor(actions, dtype=torch.long)
    next_states_t = torch.tensor(next_states, dtype=torch.long)
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    dones_t = torch.tensor(dones, dtype=torch.bool)
    t_agent.update_batch(states_t, actions_t, rewards_t, next_states_t, dones_t)
    print('Torch W[0]:', t_agent.W[0].cpu().numpy())
    print('Torch Q[0]:', t_agent.Q[0].cpu().numpy())
except Exception as e:
    print('Torch demo skipped:', e)

In [ ]:
# Safety fix: ensure at least (wait,0,0) feasible in every non-terminal state,
# then retrain Torch agent fresh with the corrected mask.
import numpy as np
import time
from tqdm import trange

# Note: This cell will skip if torch is not available
try:
    import torch
    from src.algorithms.qh_qlearning_torch import QHQLearningTorch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available - skipping torch training")

if TORCH_AVAILABLE:
    # Identify index of (wait,0,0)
    try:
        WAIT00_IDX = next(i for i,a in enumerate(ACTIONS) if a[0]=='wait' and a[1]==0 and a[2]==0)
    except StopIteration:
        WAIT00_IDX = 0

    # Rebuild feasibility mask - OPTIMIZED VERSION
    A = int(NUM_ACTIONS)
    S = int(NUM_STATES)
    mask = np.zeros((S, A), dtype=bool)
    
    # Pre-compute action feasibility criteria
    action_eat = np.array([a[1] for a in ACTIONS], dtype=np.int32)
    action_drink = np.array([a[2] for a in ACTIONS], dtype=np.int32)
    action_is_hunt = np.array([a[0] == 'hunt' for a in ACTIONS], dtype=bool)
    
    # Vectorized mask building
    print("Building feasibility mask (vectorized)...")
    start_mask = time.time()
    
    # Decode all states at once
    all_states = np.arange(S, dtype=np.int64)
    f_all, w_all, h_all, d_all = decode_batch(all_states)
    
    # For each state, check which actions are feasible
    for s_idx in range(S):
        f, w, h = f_all[s_idx], w_all[s_idx], h_all[s_idx]
        if h == 0:
            continue
        
        # Vectorized feasibility check for all actions at once
        feasible = (action_eat <= f) & (action_drink <= w)
        
        # Additional constraint for hunt actions
        hunt_feasible = (f - action_eat >= 1) & (w - action_drink >= 1)
        feasible = np.where(action_is_hunt, feasible & hunt_feasible, feasible)
        
        mask[s_idx] = feasible
        # Enforce at least wait-0-0
        mask[s_idx, WAIT00_IDX] = True
    
    print(f"Mask built in {time.time()-start_mask:.2f}s")

    # Fresh agent
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {_device}")
    
    t_agent = QHQLearningTorch(
        n_states=int(NUM_STATES),
        n_actions=int(NUM_ACTIONS),
        alpha=0.8,
        beta=0.95,
        theta_step=1e-3,
        eta_step=2e-3,
        theta_power=0.55,
        eta_power=0.52,
        step_offset=5000.0,
        init_value=0.0,
        device=_device,
    )

    num_batches = 800
    batch_size = 1024
    rng = np.random.default_rng(321)
    
    # Pre-allocate arrays for batch data - OPTIMIZATION
    states = np.empty((batch_size,), dtype=np.int64)
    actions = np.empty((batch_size,), dtype=np.int64)
    rewards = np.empty((batch_size,), dtype=np.float32)
    next_states = np.empty((batch_size,), dtype=np.int64)
    dones = np.empty((batch_size,), dtype=np.bool_)
    
    start = time.time()
    for b in trange(num_batches, desc="Torch retrain (safe mask)"):
        e = 0.2 + (0.05 - 0.2) * min(1.0, b/4000.0)
        
        # Generate batch of random states
        rng.integers(0, S, size=batch_size, out=states)
        
        # For each state, sample action and transition
        for i in range(batch_size):
            s_idx = states[i]
            feas = np.flatnonzero(mask[s_idx])
            if feas.size == 0:
                feas = np.array([WAIT00_IDX])
            
            if rng.random() < e:
                a_idx = int(rng.choice(feas))
            else:
                q_row = t_agent.Q[s_idx].detach().cpu().numpy()
                a_idx = int(feas[np.argmax(q_row[feas])])
            
            ns, r, done = sample_transition(int(s_idx), int(a_idx))
            actions[i] = a_idx
            rewards[i] = r
            next_states[i] = ns
            dones[i] = done

        t_agent.update_batch(states, actions, rewards, next_states, dones=dones, available_actions_mask=mask)

    elapsed = time.time()-start
    print(f"\nRetrain done in {elapsed:.1f}s")
    print(f"Throughput: {num_batches * batch_size / elapsed:.0f} transitions/sec")

    # Probe again
    NF, NW, NH, ND = int(N_F), int(N_W), int(N_H), int(N_D)
    _stride_w = NH * ND
    _stride_f = NW * _stride_w
    
    probe_states = [int((5*_stride_f) + (5*_stride_w) + (3*ND) + 0),
                    int((6*_stride_f) + (1*_stride_w) + (3*ND) + 4),
                    int((4*_stride_f) + (4*_stride_w) + (4*ND) + max(0, D_MAX-2))]
    for s_idx in probe_states:
        q_row = t_agent.Q[s_idx].detach().cpu().numpy()
        feasible = np.where(mask[s_idx])[0]
        best = int(feasible[np.argmax(q_row[feasible])]) if feasible.size>0 else WAIT00_IDX
        mode, eat, drink = ACTIONS[best]
        f,w,h,d = decode(int(s_idx))
        print(f"[SAFE] State (f={f},w={w},h={h},d={d}) -> best: mode={mode}, eat={eat}, drink={drink}, Q={q_row[best]:.3f}")



In [ ]:
# Torch episodic training from fixed start (f=5,w=5,h=4,d=0) with time budget
import time
import numpy as np
from tqdm import trange

# Note: This cell will skip if torch is not available
try:
    import torch
    from src.algorithms.qh_qlearning_torch import QHQLearningTorch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available - skipping torch episodic training")

if TORCH_AVAILABLE:
    # Ensure helper from earlier cell
    NF, NW, NH, ND = int(N_F), int(N_W), int(N_H), int(N_D)
    _stride_w = NH * ND
    _stride_f = NW * _stride_w

    def decode_state(s_idx: int):
        f = s_idx // _stride_f
        rem = s_idx % _stride_f
        w = rem // _stride_w
        rem2 = rem % _stride_w
        h = rem2 // ND
        d = rem2 % ND
        return int(f), int(w), int(h), int(d)

    # Build safe feasibility mask [S, A] and enforce at least (wait,0,0)
    try:
        WAIT00_IDX = next(i for i,a in enumerate(ACTIONS) if a[0]=='wait' and a[1]==0 and a[2]==0)
    except StopIteration:
        WAIT00_IDX = 0

    S = int(NUM_STATES); A = int(NUM_ACTIONS)
    
    # OPTIMIZED: Reuse mask if already computed, otherwise build it
    if 'mask' not in globals():
        print("Building feasibility mask...")
        mask = np.zeros((S, A), dtype=bool)
        
        # Pre-compute action properties
        action_eat = np.array([a[1] for a in ACTIONS], dtype=np.int32)
        action_drink = np.array([a[2] for a in ACTIONS], dtype=np.int32)
        action_is_hunt = np.array([a[0] == 'hunt' for a in ACTIONS], dtype=bool)
        
        # Decode all states
        all_states = np.arange(S, dtype=np.int64)
        f_all, w_all, h_all, d_all = decode_batch(all_states)
        
        for s_idx in range(S):
            f, w, h = f_all[s_idx], w_all[s_idx], h_all[s_idx]
            if h == 0:
                continue
            
            # Vectorized feasibility
            feasible = (action_eat <= f) & (action_drink <= w)
            hunt_feasible = (f - action_eat >= 1) & (w - action_drink >= 1)
            feasible = np.where(action_is_hunt, feasible & hunt_feasible, feasible)
            
            mask[s_idx] = feasible
            mask[s_idx, WAIT00_IDX] = True
        print("Mask built.")

    # Agent
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    t_agent = QHQLearningTorch(
        n_states=S,
        n_actions=A,
        alpha=0.8,
        beta=0.95,
        theta_step=8e-4,
        eta_step=1.6e-3,
        theta_power=0.55,
        eta_power=0.52,
        step_offset=5000.0,
        init_value=0.0,
        device=device,
    )

    # Training config
    start_state = encode(5, 5, 4, 0)
    max_steps_per_ep = D_MAX + 1
    batch_cap = 2048
    rng = np.random.default_rng(2026)

    # Epsilon schedule over episodes
    eps_start, eps_end, eps_decay_episodes = 0.2, 0.05, 400

    def eps_for_ep(ep):
        t = min(1.0, ep/eps_decay_episodes)
        return eps_start + t*(eps_end - eps_start)

    # Time budget (15 minutes)
    time_budget_sec = 15 * 60
    print(f"Starting episodic Torch QH-QL: budget={time_budget_sec}s, device={device}")

    # OPTIMIZED: Pre-allocate buffers
    buf_s = np.empty((batch_cap,), dtype=np.int64)
    buf_a = np.empty((batch_cap,), dtype=np.int64)
    buf_r = np.empty((batch_cap,), dtype=np.float32)
    buf_ns = np.empty((batch_cap,), dtype=np.int64)
    buf_d = np.empty((batch_cap,), dtype=np.bool_)
    ptr = 0

    start_ts = time.time()
    ep = 0
    steps_total = 0
    ret_ema = None

    # Pre-allocate Q row copy buffer for faster action selection
    q_row_buffer = np.empty(A, dtype=np.float32)

    while True:
        if time.time() - start_ts >= time_budget_sec:
            break
        ep += 1
        s = int(start_state)
        done = False
        ret = 0.0
        e = eps_for_ep(ep)
        
        for t in range(int(max_steps_per_ep)):
            feas = np.flatnonzero(mask[s])
            if feas.size == 0:
                a = WAIT00_IDX
            else:
                if rng.random() < e:
                    a = int(rng.choice(feas))
                else:
                    # OPTIMIZED: Reuse buffer for Q values
                    q_vals = t_agent.Q[s].detach().cpu().numpy()
                    a = int(feas[np.argmax(q_vals[feas])])
            
            ns, r, d = sample_transition(s, a)
            
            # Store in buffer
            buf_s[ptr] = s
            buf_a[ptr] = a
            buf_r[ptr] = r
            buf_ns[ptr] = ns
            buf_d[ptr] = d
            ptr += 1
            steps_total += 1
            ret += r
            s = ns
            done = d
            
            # Batch update when buffer is full
            if ptr == batch_cap:
                t_agent.update_batch(buf_s, buf_a, buf_r, buf_ns, dones=buf_d, available_actions_mask=mask)
                ptr = 0
            
            if done:
                break
        
        # Post-episode flush (if any pending)
        if ptr > 0:
            t_agent.update_batch(buf_s[:ptr], buf_a[:ptr], buf_r[:ptr], buf_ns[:ptr], dones=buf_d[:ptr], available_actions_mask=mask)
            ptr = 0
        
        ret_ema = ret if ret_ema is None else 0.9*ret_ema + 0.1*ret
        
        if ep % 10 == 0:
            elapsed = time.time() - start_ts
            print(f"ep={ep:5d} elapsed={elapsed:6.1f}s steps={steps_total:8d} eps={e:.3f} ret_ema={ret_ema:.3f}")

    elapsed = time.time() - start_ts
    print(f"\nFinished: episodes={ep}, steps={steps_total}, elapsed={elapsed:.1f}s")
    print(f"Throughput: {steps_total / elapsed:.0f} transitions/sec")

    # Probe best action at start and a couple of nearby states
    probes = [start_state,
              encode(5,5,3,0),
              encode(6,4,4,1)]
    for ps in probes:
        q_row = t_agent.Q[int(ps)].detach().cpu().numpy()
        feas = np.where(mask[int(ps)])[0]
        best = int(feas[np.argmax(q_row[feas])]) if feas.size>0 else WAIT00_IDX
        m, ef, ew = ACTIONS[best]
        f,w,h,d = decode(int(ps))
        print(f"Probe (f={f},w={w},h={h},d={d}) -> best: {m}/{ef}/{ew}, Q={q_row[best]:.3f}")



In [ ]:
# Torch episodic training from fixed start (f=5,w=5,h=4,d=0) with time budget
import time, numpy as np, torch
from tqdm import trange
from src.algorithms.qh_qlearning_torch import QHQLearningTorch

# Ensure helper from earlier cell
NF, NW, NH, ND = int(N_F), int(N_W), int(N_H), int(N_D)
_stride_w = NH * ND
_stride_f = NW * _stride_w

def decode_state(s_idx: int):
    f = s_idx // _stride_f
    rem = s_idx % _stride_f
    w = rem // _stride_w
    rem2 = rem % _stride_w
    h = rem2 // ND
    d = rem2 % ND
    return int(f), int(w), int(h), int(d)

# Build safe feasibility mask [S, A] and enforce at least (wait,0,0)
try:
    WAIT00_IDX = next(i for i,a in enumerate(ACTIONS) if a[0]=='wait' and a[1]==0 and a[2]==0)
except StopIteration:
    WAIT00_IDX = 0

S = int(NUM_STATES); A = int(NUM_ACTIONS)
mask = np.zeros((S, A), dtype=bool)
for s_idx in range(S):
    f, w, h, d = decode_state(s_idx)
    if h == 0:
        continue
    for a_idx, (mode, eat, drink) in enumerate(ACTIONS):
        feas = (eat <= f) and (drink <= w)
        if feas and mode == 'hunt':
            feas = (f - eat >= 1) and (w - drink >= 1)
        mask[s_idx, a_idx] = feas
    mask[s_idx, WAIT00_IDX] = True

# Agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t_agent = QHQLearningTorch(
    n_states=S,
    n_actions=A,
    alpha=0.8,
    beta=0.95,
    theta_step=8e-4,
    eta_step=1.6e-3,
    theta_power=0.55,
    eta_power=0.52,
    step_offset=5000.0,
    init_value=0.0,
    device=device,
)

# Training config
start_state = encode(5, 5, 4, 0)
max_steps_per_ep = D_MAX + 1
batch_cap = 2048
rng = np.random.default_rng(2026)

# Epsilon schedule over episodes
eps_start, eps_end, eps_decay_episodes = 0.2, 0.05, 400

def eps_for_ep(ep):
    t = min(1.0, ep/eps_decay_episodes)
    return eps_start + t*(eps_end - eps_start)

# Time budget (15 minutes)
time_budget_sec = 15 * 60
print(f"Starting episodic Torch QH-QL: budget={time_budget_sec}s, device={device}")

# Buffers
buf_s = np.empty((batch_cap,), dtype=np.int64)
buf_a = np.empty((batch_cap,), dtype=np.int64)
buf_r = np.empty((batch_cap,), dtype=np.float32)
buf_ns = np.empty((batch_cap,), dtype=np.int64)
buf_d = np.empty((batch_cap,), dtype=np.bool_)
ptr = 0

start_ts = time.time()
ep = 0
steps_total = 0
ret_ema = None

while True:
    if time.time() - start_ts >= time_budget_sec:
        break
    ep += 1
    s = int(start_state)
    done = False
    ret = 0.0
    e = eps_for_ep(ep)
    for t in range(int(max_steps_per_ep)):
        feas = np.flatnonzero(mask[s])
        if feas.size == 0:
            a = WAIT00_IDX
        else:
            if rng.random() < e:
                a = int(rng.choice(feas))
            else:
                q_row = t_agent.Q[s].detach().cpu().numpy()
                a = int(feas[np.argmax(q_row[feas])])
        ns, r, d = sample_transition(s, a)
        # store
        buf_s[ptr] = s; buf_a[ptr] = a; buf_r[ptr] = r; buf_ns[ptr] = ns; buf_d[ptr] = d
        ptr += 1
        steps_total += 1
        ret += r
        s = ns
        done = d
        if ptr == batch_cap:
            t_agent.update_batch(buf_s, buf_a, buf_r, buf_ns, dones=buf_d, available_actions_mask=mask)
            ptr = 0
        if done:
            break
    # post-episode flush (if any pending)
    if ptr > 0:
        t_agent.update_batch(buf_s[:ptr], buf_a[:ptr], buf_r[:ptr], buf_ns[:ptr], dones=buf_d[:ptr], available_actions_mask=mask)
        ptr = 0
    ret_ema = ret if ret_ema is None else 0.9*ret_ema + 0.1*ret
    if ep % 10 == 0:
        elapsed = time.time() - start_ts
        print(f"ep={ep:5d} elapsed={elapsed:6.1f}s steps={steps_total:8d} eps={e:.3f} ret_ema={ret_ema:.3f}")

elapsed = time.time() - start_ts
print(f"Finished: episodes={ep}, steps={steps_total}, elapsed={elapsed:.1f}s")

# Probe best action at start and a couple of nearby states
probes = [start_state,
          encode(5,5,3,0),
          encode(6,4,4,1)]
for ps in probes:
    q_row = t_agent.Q[int(ps)].detach().cpu().numpy()
    feas = np.where(mask[int(ps)])[0]
    best = int(feas[np.argmax(q_row[feas])]) if feas.size>0 else WAIT00_IDX
    m, ef, ew = ACTIONS[best]
    f,w,h,d = decode_state(int(ps))
    print(f"Probe (f={f},w={w},h={h},d={d}) -> best: {m}/{ef}/{ew}, Q={q_row[best]:.3f}")
